In [2]:
import requests
from dotenv import load_dotenv
import os

In [7]:
load_dotenv(".env")
KNMI_EDR_API_KEY = os.environ["KNMI_EDR_API_KEY"]
KNMI_OPEN_DATA_KEY = os.environ["KNMI_OPEN_DATA_KEY"]

In [4]:
headers = {"Authorization": KNMI_EDR_API_KEY}

r = requests.get("https://api.dataplatform.knmi.nl/edr/v1/collections", headers=headers)
print(r.status_code)
collections = r.json()
for c in collections.get("collections", []):
    print(c["id"], "-", c.get("title"))

200
Tg1 - Temperature - gridded daily mean temperature in the Netherlands
Tn1 - Temperature - gridded daily minimum temperature in the Netherlands
Tx1 - Temperature - gridded daily maximum temperature in the Netherlands
Rd1 - Precipitation - gridded daily precipitation sum in the Netherlands
EV24 - Evaporation - gridded daily Makkink evaporation in the Netherlands.
wins50 - WINS50 - wind at 10-600 meter height for the Netherlands from HARMONIE-AROME as daily files
10-minute-in-situ-meteorological-observations - Near real-time 10-minute automated in-situ ground-based meteorological observations in the Netherlands
daily-in-situ-meteorological-observations - Daily and automated in-situ ground-based meteorological observations in the Netherlands
daily-in-situ-meteorological-observations-validated - Daily, validated and automated in-situ ground-based meteorological observations in the Netherlands
hourly-in-situ-meteorological-observations - Hourly and automated in-situ ground-based meteorol

In [5]:
collection = "wins50"
base_url = f"https://api.dataplatform.knmi.nl/edr/v1/collections/{collection}"

r = requests.get(base_url, headers=headers)
data = r.json()

print("Temporal extent:", data.get("extent", {}).get("temporal", {}).get("interval"))
print()
print("Parameter names:")
for name, info in data.get("parameter_names", {}).items():
    print(f"  {name}: {info.get('unit', {}).get('label')}")

Temporal extent: [['2019-01-01T00:00:00Z', '2022-01-01T00:00:00Z']]

Parameter names:
  wdir: None
  wspeed: None
  ta: None
  p: None
  hur: None


In [11]:
headers = {"Authorization": KNMI_OPEN_DATA_KEY}

dataset_name = "harmonie_arome_cy43_p2b"
dataset_version = "1.0"

r = requests.get(
    f"https://api.dataplatform.knmi.nl/open-data/v1/datasets/{dataset_name}/versions/{dataset_version}/files",
    headers=headers,
    params={"maxKeys": 5, "orderBy": "created", "sorting": "desc"}
)
print(r.status_code)
print(r.json())

200
{'isTruncated': True, 'resultCount': 5, 'files': [{'filename': 'harm43_v1_P2b_2026061712.tar', 'size': 1450096640, 'created': '2026-06-17T14:29:41+00:00', 'lastModified': '2026-06-17T14:29:41+00:00'}, {'filename': 'harm43_v1_P2b_2026061711.tar', 'size': 1450096640, 'created': '2026-06-17T13:30:18+00:00', 'lastModified': '2026-06-17T13:30:18+00:00'}, {'filename': 'harm43_v1_P2b_2026061710.tar', 'size': 1450096640, 'created': '2026-06-17T12:29:43+00:00', 'lastModified': '2026-06-17T12:29:43+00:00'}, {'filename': 'harm43_v1_P2b_2026061709.tar', 'size': 1450096640, 'created': '2026-06-17T11:33:53+00:00', 'lastModified': '2026-06-17T11:33:53+00:00'}, {'filename': 'harm43_v1_P2b_2026061708.tar', 'size': 1450096640, 'created': '2026-06-17T10:30:16+00:00', 'lastModified': '2026-06-17T10:30:16+00:00'}], 'maxResults': 5, 'startAfterFilename': '', 'nextPageToken': 'eyJjcmVhdGVkIjogIjIwMjYtMDYtMTdUMTA6MzA6MTYrMDA6MDAiLCAiaWQiOiAiaGFybW9uaWVfYXJvbWVfY3k0M19wMmJfMS4wX2hhcm00M192MV9QMmJfMjAyNjA2M

In [2]:
import requests
import pandas as pd

def fetch_openmeteo_forecast(lat: float, lon: float, height_m: int = 120, forecast_days: int = 7) -> pd.DataFrame:
    """
    Fetches hourly wind speed forecast for a given location and height.
    height_m should be one of: 10, 80, 120, 180 (nearest available to hub height).
    """
    wind_param = f"wind_speed_{height_m}m"

    response = requests.get(
        "https://api.open-meteo.com/v1/forecast",
        params={
            "latitude": lat,
            "longitude": lon,
            "hourly": wind_param,
            "wind_speed_unit": "ms",
            "forecast_days": forecast_days,
            "timezone": "UTC",
        }
    )
    response.raise_for_status()
    data = response.json()

    df = pd.DataFrame({
        "timestamp": pd.to_datetime(data["hourly"]["time"]),
        "wind_speed_ms": data["hourly"][wind_param],
    })
    return df

In [3]:
farms = {
    "Borssele_12":          {"lat": 51.75, "lon": 3.25, "platform_height": 116.5},
    "Borssele_34":          {"lat": 51.72, "lon": 3.22, "platform_height": 100},
    "Gemini":               {"lat": 54.03, "lon": 5.95, "platform_height": 120},
    "Hollandse_Kust_Zuid":  {"lat": 52.35, "lon": 4.25, "platform_height": 125.5},
    "Hollandse_Kust_Noord": {"lat": 52.75, "lon": 4.50, "platform_height": 125.5},
}

available_heights = [10, 80, 120, 180]

def nearest_height(target: float) -> int:
    return min(available_heights, key=lambda h: abs(h - target))

forecasts = {}
for farm_id, info in farms.items():
    height = nearest_height(info["platform_height"])
    forecasts[farm_id] = fetch_openmeteo_forecast(info["lat"], info["lon"], height_m=height)
    print(f"{farm_id}: fetched at {height}m, {len(forecasts[farm_id])} rows")

Borssele_12: fetched at 120m, 168 rows
Borssele_34: fetched at 80m, 168 rows
Gemini: fetched at 120m, 168 rows
Hollandse_Kust_Zuid: fetched at 120m, 168 rows
Hollandse_Kust_Noord: fetched at 120m, 168 rows


In [4]:
forecasts

{'Borssele_12':               timestamp  wind_speed_ms
 0   2026-06-18 00:00:00          10.41
 1   2026-06-18 01:00:00           9.80
 2   2026-06-18 02:00:00           8.47
 3   2026-06-18 03:00:00           8.57
 4   2026-06-18 04:00:00           7.76
 ..                  ...            ...
 163 2026-06-24 19:00:00           6.94
 164 2026-06-24 20:00:00           6.88
 165 2026-06-24 21:00:00           6.78
 166 2026-06-24 22:00:00           6.64
 167 2026-06-24 23:00:00           6.54
 
 [168 rows x 2 columns],
 'Borssele_34':               timestamp  wind_speed_ms
 0   2026-06-18 00:00:00           9.94
 1   2026-06-18 01:00:00           9.36
 2   2026-06-18 02:00:00           8.09
 3   2026-06-18 03:00:00           8.09
 4   2026-06-18 04:00:00           7.21
 ..                  ...            ...
 163 2026-06-24 19:00:00           6.63
 164 2026-06-24 20:00:00           6.60
 165 2026-06-24 21:00:00           6.58
 166 2026-06-24 22:00:00           6.51
 167 2026-06-24 23:00:0

In [4]:
import requests
import pandas as pd
from datetime import datetime, timedelta

ENTSOE_WEB_URL = "https://transparency.entsoe.eu/generation/actual/perType/generation/load"

HEADERS = {
    "accept": "application/json",
    "content-type": "application/json; charset=utf-8",
    "cookie": "uu.app.bpl=0",
    "origin": "https://transparency.entsoe.eu",
}

# PSR production type codes (subset relevant to this project)
PRODUCTION_TYPE_WIND_OFFSHORE = "B18"


def fetch_entsoe_generation(area_code: str, date_from: str, date_to: str, timezone: str = "CET") -> dict:
    """
    Fetch actual generation per production type from the ENTSOE Transparency
    Platform's public website endpoint (no security token required).

    area_code: e.g. "BZN|10YNL----------L" for the Netherlands bidding zone.
    date_from, date_to: ISO date strings, e.g. "2026-06-17".
    Returns the raw parsed JSON response.
    """
    payload = {
        "dateTimeRange": {
            "from": f"{date_from}T22:00:00.000Z",
            "to": f"{date_to}T22:00:00.000Z",
        },
        "areaList": [area_code],
        "timeZone": timezone,
        "sorterList": [],
        "filterMap": {},
    }

    response = requests.post(ENTSOE_WEB_URL, headers=HEADERS, json=payload)
    response.raise_for_status()
    return response.json()


def extract_production_type_series(data: dict, production_type: str, resample_hourly: bool = False) -> pd.Series:
    """
    Extract a single production type's generation series from the raw
    fetch_entsoe_generation() response, indexed by timestamp (UTC).

    If resample_hourly is True, the native 15-minute series is averaged
    into hourly buckets to match a typically hourly Timeseries collection.
    """
    for instance in data.get("instanceList", []):
        if instance["businessDimensionMap"].get("PRODUCTION_TYPE") != production_type:
            continue

        for period in instance["curveData"]["periodList"]:
            start = datetime.fromisoformat(period["timeInterval"]["from"].replace("Z", "+00:00"))
            resolution_minutes = 15  # PT15M

            timestamps = []
            values = []
            for idx_str, point in sorted(period["pointMap"].items(), key=lambda kv: int(kv[0])):
                idx = int(idx_str)
                ts = start + timedelta(minutes=resolution_minutes * idx)

                value = point[0]
                if isinstance(value, dict):
                    # missing/not-yet-available point, e.g. {"alt": "-"} or {"alt": "n/e"}
                    continue

                timestamps.append(ts)
                values.append(float(value))

            series = pd.Series(values, index=pd.DatetimeIndex(timestamps), name=production_type)

            if resample_hourly:
                series = series.resample("h").mean()

            return series

    raise ValueError(f"Production type '{production_type}' not found in response")


def insert_entsoe_offshore_wind_actual(area_code: str, date_from: str, date_to: str,
                                        mongo_uri: str, mongo_db: str = "wind_farm",
                                        collection_name: str = "Timeseries") -> None:
    """
    Fetch actual offshore wind generation (B19) for the given area and date
    range from the ENTSOE web endpoint, resample to hourly, convert from UTC
    to Europe/Amsterdam local time (matching the rest of the Timeseries
    collection), and upsert it into actual_mw on the "TOTAL" pseudo-farm
    documents.
    """
    from pymongo import MongoClient, UpdateOne

    data = fetch_entsoe_generation(area_code, date_from, date_to)
    series = extract_production_type_series(data, PRODUCTION_TYPE_WIND_OFFSHORE, resample_hourly=True)

    # series index is UTC (tz-aware); convert to local Dutch time to match
    # the rest of the collection (KNMI/Open-Meteo data is stored in
    # Europe/Amsterdam local time, not UTC).
    series.index = series.index.tz_convert("Europe/Amsterdam").tz_localize(None)

    client = MongoClient(mongo_uri)
    collection = client[mongo_db][collection_name]

    operations = [
        UpdateOne(
            {"timestamp": ts.to_pydatetime(), "farm_id": "TOTAL"},
            {"$set": {"actual_mw": float(value)}},
            upsert=True
        )
        for ts, value in series.items()
        if pd.notna(value)
    ]

    if not operations:
        print("No data points to insert")
        return

    result = collection.bulk_write(operations, ordered=False)
    print(f"TOTAL actual_mw: upserted={result.upserted_count}, modified={result.modified_count}")


if __name__ == "__main__":
    data = fetch_entsoe_generation(
        area_code="BZN|10YNL----------L",
        date_from="2026-06-20",
        date_to="2026-06-21",
    )

    offshore_wind = extract_production_type_series(data, PRODUCTION_TYPE_WIND_OFFSHORE)
    print(offshore_wind)

2026-06-20 22:00:00+00:00     197.86
2026-06-20 22:15:00+00:00     221.76
2026-06-20 22:30:00+00:00     243.68
2026-06-20 22:45:00+00:00     256.01
2026-06-20 23:00:00+00:00     276.84
                              ...   
2026-06-21 18:15:00+00:00    1980.51
2026-06-21 18:30:00+00:00    1944.13
2026-06-21 18:45:00+00:00    1977.85
2026-06-21 19:00:00+00:00    1985.55
2026-06-21 19:15:00+00:00    1976.45
Name: B18, Length: 86, dtype: float64


In [2]:
offshore_wind

2026-06-20 22:00:00+00:00     11.36
2026-06-20 22:15:00+00:00     16.99
2026-06-20 22:30:00+00:00     24.30
2026-06-20 22:45:00+00:00     27.07
2026-06-20 23:00:00+00:00     37.75
                              ...  
2026-06-21 15:45:00+00:00    244.69
2026-06-21 16:00:00+00:00    253.21
2026-06-21 16:15:00+00:00    256.95
2026-06-21 16:30:00+00:00    253.54
2026-06-21 16:45:00+00:00    252.76
Name: B19, Length: 76, dtype: float64